# LeetCode 3300 — Minimum Element After Replacement With Digit Sum bằng Python

Link đề: https://leetcode.com/problems/minimum-element-after-replacement-with-digit-sum/ · Độ khó: **Easy** · Chủ đề: Array, Math

Skeleton LeetCode:

```python
class Solution:
    def minElement(self, nums: List[int]) -> int:
        ...
```

Ghi chú: mọi đáp án trong notebook đã được kiểm chứng bằng code (kể cả vét cạn **mọi số từ `1` đến `10 000`** và hàng nghìn mảng ngẫu nhiên đúng ràng buộc). Ba ví dụ chính lấy từ đề; ví dụ nào tự dựng thêm có ghi rõ "tự dựng" — nếu muốn khớp từng chữ thì nên đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: thay **mỗi** phần tử của `nums` bằng **tổng các chữ số** của nó, rồi trả về **phần tử nhỏ nhất** sau khi thay hết.
2. Nắm các công cụ Python: đổi số sang chuỗi bằng **`str`**, duyệt từng **ký tự** của chuỗi, đổi ký tự sang số bằng **`int`**, **`sum`** và **`map`**, tách chữ số bằng **`% 10`** và **`// 10`** (hay `divmod`), vòng **`while`**, và hàm **`min`** (kể cả với biểu thức sinh *generator*).
3. Đi từ cách **dựng cả mảng mới rồi tìm min** → **đổi chuỗi** → **tách chữ số bằng `%`/`//`** → **một lượt, giữ min tạm** → **bảng tra tổng chữ số dựng bằng quy hoạch động** `ds[x] = ds[x // 10] + x % 10`.
4. Hiểu mấu chốt: bài gồm **hai việc độc lập** — (a) tính tổng chữ số của một số; (b) lấy giá trị nhỏ nhất của các kết quả. **Phải thay hết rồi mới lấy min**: phần tử nhỏ nhất ban đầu chưa chắc có tổng chữ số nhỏ nhất (ví dụ `19 -> 10` nhưng `100 -> 1`).
5. Tránh các lỗi kinh điển: lấy tổng chữ số của **số nhỏ nhất** thay vì **nhỏ nhất của các tổng**, `sum(str(x))` (cộng chuỗi với `0` gây `TypeError`), quên `x //= 10` (vòng lặp **vô hạn**), dùng `/` thay `//`, và khởi tạo `best = 0` (min không bao giờ cập nhật được).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`.

Bạn thay **mỗi** phần tử trong `nums` bằng **tổng các chữ số** của chính nó.

Trả về phần tử **nhỏ nhất** trong `nums` sau khi đã thay thế **tất cả**.

**Ví dụ 1**

- Input: `nums = [10,12,13,14]`
- Output: `1`
- Giải thích: sau khi thay, `nums` thành `[1, 3, 4, 5]`, phần tử nhỏ nhất là `1`.

**Ví dụ 2**

- Input: `nums = [1,2,3,4]`
- Output: `1`
- Giải thích: `nums` thành `[1, 2, 3, 4]` (số một chữ số giữ nguyên), nhỏ nhất là `1`.

**Ví dụ 3**

- Input: `nums = [999,19,199]`
- Output: `10`
- Giải thích: `nums` thành `[27, 10, 19]`, nhỏ nhất là `10`.

**Ràng buộc:** `1 <= nums.length <= 100`; `1 <= nums[i] <= 10^4`.

In [1]:
import random
from typing import List

# (nums, kết quả kỳ vọng) — ba ví dụ lấy từ đề
examples = [
    ([10, 12, 13, 14], 1),
    ([1, 2, 3, 4], 1),
    ([999, 19, 199], 10),
]
for nums, expected in examples:
    print(f"nums={nums} -> kỳ vọng {expected}")

nums=[10, 12, 13, 14] -> kỳ vọng 1
nums=[1, 2, 3, 4] -> kỳ vọng 1
nums=[999, 19, 199] -> kỳ vọng 10


## 3. Kiến thức nền

### 3.1. Đổi số thành chuỗi và duyệt từng chữ số

`str(1234)` cho chuỗi `"1234"`. Chuỗi là dãy ký tự nên ta duyệt được bằng `for ch in "1234":`; mỗi `ch` là **một ký tự** (`"1"`, `"2"`, ...), vẫn là **chuỗi** chứ chưa phải số.

In [2]:
x = 1234
s = str(x)
print(s, type(s), len(s))                # 1234 <class 'str'> 4
for ch in s:
    print(repr(ch), type(ch).__name__, end=" | ")
print()

1234 <class 'str'> 4
'1' str | '2' str | '3' str | '4' str | 


### 3.2. `int` đổi ký tự về số; `sum` và `map`

`int("7")` cho số `7`. `sum([1, 2, 3])` cộng các phần tử số của danh sách. `map(int, "1234")` áp `int` lên từng ký tự, sinh các số `1, 2, 3, 4` (ta bọc `list(...)` để xem). Do đó **tổng chữ số** của `1234` là `sum(map(int, str(1234)))`.

Chú ý: `sum("1234")` **lỗi** vì `sum` bắt đầu từ `0` rồi cộng thêm chuỗi (`0 + "1"` không hợp lệ).

In [3]:
print(int("7") + 1)                                 # 8
print(list(map(int, "1234")))                        # [1, 2, 3, 4]
print(sum(map(int, str(1234))))                      # 10
try:
    print(sum("1234"))
except TypeError as e:
    print("TypeError:", e)

8
[1, 2, 3, 4]
10
TypeError: unsupported operand type(s) for +: 'int' and 'str'


### 3.3. `% 10` và `// 10`: tách chữ số bằng toán

- `x % 10` là **chữ số cuối** (chữ số hàng đơn vị) của `x`: `1234 % 10 = 4`.
- `x // 10` là `x` **bỏ chữ số cuối**: `1234 // 10 = 123`.
- `divmod(x, 10)` trả cả hai cùng lúc: `(123, 4)`.

Lặp: lấy chữ số cuối, rồi bỏ nó, cho tới khi `x` về `0`. Thứ tự lấy chữ số là từ **phải sang trái** — với bài cộng tổng thì thứ tự không quan trọng.

In [4]:
x = 1234
print(x % 10, x // 10, divmod(x, 10))        # 4 123 (123, 4)
digits = []
while x > 0:
    x, d = divmod(x, 10)
    digits.append(d)
print(digits)                                 # [4, 3, 2, 1]  -- từ phải sang trái

4 123 (123, 4)
[4, 3, 2, 1]


### 3.4. Vòng `while` và điều kiện dừng

`while điều_kiện:` lặp cho đến khi điều kiện sai. Bắt buộc **thay đổi** biến trong điều kiện ở mỗi vòng, nếu không vòng lặp **không bao giờ dừng**. Trong bài này điều kiện là `x > 0` và ta phải cập nhật `x = x // 10` (hay `x //= 10`) ở mỗi vòng.

In [5]:
x = 305
steps = 0
while x > 0:
    x //= 10            # BẮT BUỘC: nếu quên dòng này, vòng lặp chạy mãi
    steps += 1
print(steps)            # 3  -- số 305 có 3 chữ số

3


### 3.5. `/` và `//`: số thực và chia nguyên

`/` luôn cho **số thực** (`7 / 2 = 3.5`, kể cả `10 / 5 = 2.0`). `//` là **chia nguyên** (`7 // 2 = 3`). Khi tách chữ số ta cần `//`; nếu nhầm `/` thì `x` thành số thực và **không bao giờ đúng bằng `0`** trong thời gian hợp lý (xem Mục 13).

In [6]:
print(7 / 2, 7 // 2, 10 / 5, 10 // 5)        # 3.5 3 2.0 2
print(type(10 / 5).__name__, type(10 // 5).__name__)   # float int

3.5 3 2.0 2
float int


### 3.6. `min` và generator expression

`min([5, 3, 8])` cho `3`. Có thể đưa thẳng một **generator** (biểu thức trong ngoặc, không tạo danh sách trung gian): `min(f(x) for x in nums)` tính `f(x)` cho từng phần tử rồi lấy nhỏ nhất. Ngoài ra khi tự tìm min bằng vòng lặp, **giá trị khởi tạo** phải là phần tử đầu tiên hoặc một số **rất lớn** (`float("inf")`), không phải `0`.

In [7]:
nums = [27, 10, 19]
print(min(nums))                                            # 10
print(min(x // 2 for x in nums))                            # 5 -- generator, không tạo list
print(min(nums, default=None), min([], default=None))        # 10 None
best = float("inf")
for x in nums:
    if x < best:
        best = x
print(best)                                                 # 10

10
5
10 None
10


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Bài gồm hai bước tách biệt:

1. **Thay** mỗi số bằng tổng chữ số của nó.
2. **Lấy min** của danh sách mới.

Tính tay ví dụ 3: `nums = [999, 19, 199]`.

| số | các chữ số | tổng chữ số |
|---|---|---|
| `999` | 9, 9, 9 | 27 |
| `19` | 1, 9 | 10 |
| `199` | 1, 9, 9 | 19 |

Danh sách mới `[27, 10, 19]`, nhỏ nhất `10`.

**Cạm bẫy:** đừng lấy min **trước** rồi mới tính tổng chữ số. Ví dụ `[19, 100]`: số nhỏ nhất là `19` với tổng chữ số `10`; nhưng `100` có tổng chữ số `1` nhỏ hơn nhiều. Đáp án đúng là `1`, không phải `10`.

In [8]:
for nums in ([999, 19, 199], [19, 100]):
    sums = [sum(int(ch) for ch in str(x)) for x in nums]
    print(f"nums = {nums} -> tổng chữ số {sums} -> min = {min(sums)}")

nums = [999, 19, 199] -> tổng chữ số [27, 10, 19] -> min = 10
nums = [19, 100] -> tổng chữ số [10, 1] -> min = 1


## 5. Cách 1 — Brute Force / baseline: dựng cả mảng mới rồi tìm min bằng vòng lặp

**Ý tưởng:** làm đúng như đề mô tả. Bước 1: viết hàm `digit_sum_str(x)` đổi `x` thành chuỗi, cộng từng chữ số. Bước 2: tạo danh sách `replaced` gồm tổng chữ số của từng phần tử. Bước 3: tự tìm giá trị nhỏ nhất bằng vòng lặp (giữ `best`, khởi tạo bằng phần tử đầu).

Dễ hiểu, từng việc một, nhưng **tốn thêm danh sách** và nhiều dòng.

In [9]:
def digit_sum_str(x: int) -> int:
    total = 0
    for ch in str(x):          # mỗi ch là một ký tự chữ số
        total += int(ch)
    return total


def min_element_baseline(nums: List[int]) -> int:
    replaced = []
    for x in nums:
        replaced.append(digit_sum_str(x))      # bước 1: thay từng số bằng tổng chữ số
    best = replaced[0]
    for v in replaced:                         # bước 2: tự tìm min
        if v < best:
            best = v
    return best


print(digit_sum_str(1234), digit_sum_str(999), digit_sum_str(10000))   # 10 27 1
for nums, expected in examples:
    print(min_element_baseline(nums), "kỳ vọng", expected)

10 27 1
1 kỳ vọng 1
1 kỳ vọng 1
10 kỳ vọng 10


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)` và `d` là số chữ số của phần tử (`d <= 5`). Mỗi phần tử tốn `O(d)` để tính tổng chữ số ⇒ **Time `O(n · d)`**; vì `d <= 5` là hằng số nên coi là **`O(n)`**. Dựng danh sách `replaced` dài `n` ⇒ **Space `O(n)`**. **Nút thắt:** không cần lưu cả danh sách `replaced` — ta chỉ cần giá trị nhỏ nhất, có thể cập nhật `best` ngay khi tính xong từng tổng.

## 6. Ý tưởng thuật toán / bước đột phá

Hai cải tiến nhỏ nhưng đáng giá:

1. **Không lưu cả mảng mới.** Với mỗi `x`, tính tổng chữ số `s` rồi so ngay với `best` (min tạm thời): `best = min(best, s)`. Bộ nhớ phụ chỉ là vài biến.
2. **Tổng chữ số bằng toán.** `x % 10` lấy chữ số cuối, `x //= 10` bỏ nó — không cần đổi sang chuỗi và `int` từng ký tự (nhanh hơn và là kỹ thuật dùng lại ở nhiều bài khác).

Thuật toán: `best = vô cùng`; với mỗi `x` trong `nums`: tính `s = digit_sum(x)`; `best = min(best, s)`; cuối cùng trả `best`.

In [10]:
nums = [999, 19, 199]
best = float("inf")
for x in nums:
    s = 0
    y = x
    while y > 0:
        s += y % 10
        y //= 10
    best = min(best, s)
    print(f"x = {x}: tổng chữ số = {s}, best = {best}")
print(best)                 # 10

x = 999: tổng chữ số = 27, best = 27
x = 19: tổng chữ số = 10, best = 10
x = 199: tổng chữ số = 19, best = 10
10


## 7. Các cách tiếp theo

### Cách 2 — Tổng chữ số bằng `% 10` và `// 10` (hàm riêng)

Giải thích từng đoạn:

- `while x > 0`: còn chữ số để xử lý.
- `total += x % 10`: cộng chữ số cuối.
- `x //= 10`: bỏ chữ số cuối (đây là bước làm vòng lặp **kết thúc**).
- Hàm `min_element` gọi `digit_sum_math` cho từng phần tử và cập nhật `best`.

In [11]:
def digit_sum_math(x: int) -> int:
    total = 0
    while x > 0:
        total += x % 10
        x //= 10
    return total


def min_element_math(nums: List[int]) -> int:
    best = float("inf")
    for x in nums:
        best = min(best, digit_sum_math(x))
    return best


class Solution2:
    def minElement(self, nums: List[int]) -> int:
        best = digit_sum_math(nums[0])
        for x in nums[1:]:
            best = min(best, digit_sum_math(x))
        return best


print(digit_sum_math(1234), digit_sum_math(999), digit_sum_math(10000))
for nums, expected in examples:
    print(min_element_math(nums), Solution2().minElement(nums), "kỳ vọng", expected)

10 27 1
1 1 kỳ vọng 1
1 1 kỳ vọng 1
10 10 kỳ vọng 10


**Độ phức tạp Cách 2:** **Time `O(n · d)` ≈ `O(n)`** (`d <= 5`), **Space `O(1)`**.

### Cách 3 — Một dòng: `min` + generator + `sum(map(int, str(x)))`

Gộp ba ý: `str(x)` (chuỗi chữ số), `map(int, ...)` (đổi từng ký tự sang số), `sum(...)` (tổng chữ số), và `min(... for x in nums)` (lấy nhỏ nhất). Không tạo danh sách trung gian.

In [12]:
def min_element_oneliner(nums: List[int]) -> int:
    return min(sum(map(int, str(x))) for x in nums)


class Solution3:
    def minElement(self, nums: List[int]) -> int:
        return min(sum(int(ch) for ch in str(x)) for x in nums)


for nums, expected in examples:
    print(min_element_oneliner(nums), Solution3().minElement(nums), "kỳ vọng", expected)

1 1 kỳ vọng 1
1 1 kỳ vọng 1
10 10 kỳ vọng 10


**Độ phức tạp Cách 3:** **Time `O(n · d)` ≈ `O(n)`**, **Space `O(1)`** (generator không lưu danh sách; chuỗi `str(x)` chỉ dài tối đa `5`).

### Cách 4 — Một lượt, `divmod`, giữ `best` và thoát sớm khi `best == 1`

Vì mỗi số nguyên dương có tổng chữ số **`>= 1`**, nếu đã tìm được tổng `1` thì **không thể tốt hơn** — dừng ngay. Đây là một tối ưu nhỏ (không đổi độ phức tạp tệ nhất) nhưng minh họa cách nghĩ "cận dưới". Dùng `divmod(x, 10)` để lấy cả thương và dư trong một lệnh.

In [13]:
def min_element_early(nums: List[int]) -> int:
    best = float("inf")
    for x in nums:
        s = 0
        while x > 0:
            x, d = divmod(x, 10)
            s += d
        if s < best:
            best = s
        if best == 1:            # tổng chữ số của số nguyên dương luôn >= 1 -> không thể nhỏ hơn
            break
    return best


class Solution4:
    def minElement(self, nums: List[int]) -> int:
        best = 10 ** 9
        for x in nums:
            s = 0
            while x:
                x, d = divmod(x, 10)
                s += d
            best = min(best, s)
            if best == 1:
                break
        return best


for nums, expected in examples:
    print(min_element_early(nums), Solution4().minElement(nums), "kỳ vọng", expected)

1 1 kỳ vọng 1
1 1 kỳ vọng 1
10 10 kỳ vọng 10


Ở `Solution4`, `while x:` tương đương `while x > 0:` vì `0` là "sai" trong Python; `best = 10 ** 9` là một số **lớn hơn mọi tổng chữ số có thể** (tối đa `36`) thay cho `float("inf")` — giữ `best` luôn là số nguyên.

**Độ phức tạp Cách 4:** **Time `O(n · d)` ≈ `O(n)`** tệ nhất, thường nhanh hơn nhờ thoát sớm; **Space `O(1)`**.

### Cách 5 — Bảng tra tổng chữ số bằng quy hoạch động: `ds[x] = ds[x // 10] + x % 10`

Quan sát: tổng chữ số của `x` = tổng chữ số của `x // 10` (bỏ chữ số cuối) **cộng** chữ số cuối `x % 10`. Vì `x <= 10⁴`, ta dựng sẵn bảng `ds` cho mọi `0..10 000` bằng công thức này rồi tra `ds[x]` trong `O(1)`. Cách này đắt khi chỉ dùng vài lần, nhưng **rất hữu ích khi có nhiều truy vấn** hoặc nhiều test; ở LeetCode thì bảng dựng lại mỗi lần gọi nên ta dựng một lần ở mức module.

In [14]:
LIMIT = 10 ** 4
ds = [0] * (LIMIT + 1)
for v in range(1, LIMIT + 1):
    ds[v] = ds[v // 10] + v % 10          # tổng chữ số của v = tổng chữ số của v // 10 + chữ số cuối


def min_element_table(nums: List[int]) -> int:
    return min(ds[x] for x in nums)


class Solution5:
    def minElement(self, nums: List[int]) -> int:
        table = [0] * (max(nums) + 1)         # chỉ cần tới max(nums)
        for v in range(1, len(table)):
            table[v] = table[v // 10] + v % 10
        return min(table[x] for x in nums)


print(ds[0], ds[1], ds[19], ds[999], ds[10000])         # 0 1 10 27 1
for nums, expected in examples:
    print(min_element_table(nums), Solution5().minElement(nums), "kỳ vọng", expected)

0 1 10 27 1
1 1 kỳ vọng 1
1 1 kỳ vọng 1
10 10 kỳ vọng 10


**Độ phức tạp Cách 5:** dựng bảng `O(V)` với `V = max(nums) <= 10⁴`, mỗi lần tra `O(1)` ⇒ **Time `O(V + n)`**, **Space `O(V)`**. Với đầu vào nhỏ (`n <= 100`) cách này **không** nhanh hơn Cách 2/3, nhưng đáng biết khi phải tính tổng chữ số cho **hàng triệu** truy vấn trong một khoảng giá trị nhỏ.

## 8. TRACE từng bước

Ta trace Cách 2 (`% 10`, `// 10`) trên `x = 199`, rồi cả bài với `nums = [999, 19, 199]`, rồi Cách 5 (bảng) cho vài giá trị đầu.

In [15]:
def trace_digit_sum(x: int) -> int:
    print(f"tổng chữ số của {x}")
    original = x
    total = 0
    step = 0
    while x > 0:
        step += 1
        d = x % 10
        total += d
        print(f"  vòng {step}: x = {x}, x % 10 = {d}, total = {total}, x // 10 = {x // 10}")
        x //= 10
    print(f"  x = 0 -> dừng. Tổng chữ số của {original} = {total}")
    return total


assert trace_digit_sum(199) == 19
assert trace_digit_sum(10) == 1

tổng chữ số của 199
  vòng 1: x = 199, x % 10 = 9, total = 9, x // 10 = 19
  vòng 2: x = 19, x % 10 = 9, total = 18, x // 10 = 1
  vòng 3: x = 1, x % 10 = 1, total = 19, x // 10 = 0
  x = 0 -> dừng. Tổng chữ số của 199 = 19
tổng chữ số của 10
  vòng 1: x = 10, x % 10 = 0, total = 0, x // 10 = 1
  vòng 2: x = 1, x % 10 = 1, total = 1, x // 10 = 0
  x = 0 -> dừng. Tổng chữ số của 10 = 1


In [16]:
def trace_min_element(nums: List[int]) -> int:
    print(f"nums = {nums}")
    best = float("inf")
    for idx, x in enumerate(nums):
        s, y = 0, x
        while y > 0:
            s += y % 10
            y //= 10
        old = best
        best = min(best, s)
        print(f"  phần tử {idx}: x = {x} -> tổng chữ số {s}; best: {old} -> {best}")
    return best


assert trace_min_element([999, 19, 199]) == 10
assert trace_min_element([19, 100]) == 1

nums = [999, 19, 199]
  phần tử 0: x = 999 -> tổng chữ số 27; best: inf -> 27
  phần tử 1: x = 19 -> tổng chữ số 10; best: 27 -> 10
  phần tử 2: x = 199 -> tổng chữ số 19; best: 10 -> 10
nums = [19, 100]
  phần tử 0: x = 19 -> tổng chữ số 10; best: inf -> 10
  phần tử 1: x = 100 -> tổng chữ số 1; best: 10 -> 1


In [17]:
def trace_table(limit: int):
    ds_local = [0] * (limit + 1)
    for v in range(1, limit + 1):
        ds_local[v] = ds_local[v // 10] + v % 10
        print(f"  ds[{v}] = ds[{v // 10}] + {v % 10} = {ds_local[v]}")
    return ds_local


print("dựng bảng cho 0..21")
assert trace_table(21)[19] == 10

dựng bảng cho 0..21
  ds[1] = ds[0] + 1 = 1
  ds[2] = ds[0] + 2 = 2
  ds[3] = ds[0] + 3 = 3
  ds[4] = ds[0] + 4 = 4
  ds[5] = ds[0] + 5 = 5
  ds[6] = ds[0] + 6 = 6
  ds[7] = ds[0] + 7 = 7
  ds[8] = ds[0] + 8 = 8
  ds[9] = ds[0] + 9 = 9
  ds[10] = ds[1] + 0 = 1
  ds[11] = ds[1] + 1 = 2
  ds[12] = ds[1] + 2 = 3
  ds[13] = ds[1] + 3 = 4
  ds[14] = ds[1] + 4 = 5
  ds[15] = ds[1] + 5 = 6
  ds[16] = ds[1] + 6 = 7
  ds[17] = ds[1] + 7 = 8
  ds[18] = ds[1] + 8 = 9
  ds[19] = ds[1] + 9 = 10
  ds[20] = ds[2] + 0 = 2
  ds[21] = ds[2] + 1 = 3


Bảng trace cả bài với `nums = [999, 19, 199]` (Cách 2):

| phần tử | `x` | tổng chữ số | `best` trước | `best` sau |
|---|---|---|---|---|
| 0 | `999` | 27 | `inf` | 27 |
| 1 | `19` | 10 | 27 | 10 |
| 2 | `199` | 19 | 10 | 10 |

Kết quả `10`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (tách chữ số bằng toán, một lượt giữ `best`): không cần đổi chuỗi, bộ nhớ `O(1)`, dễ dò từng bước.

In [18]:
class Solution:
    def minElement(self, nums: List[int]) -> int:
        best = 10 ** 9                       # lớn hơn mọi tổng chữ số có thể (tối đa 36)
        for x in nums:
            s = 0
            while x > 0:
                s += x % 10                  # cộng chữ số cuối
                x //= 10                     # bỏ chữ số cuối
            if s < best:
                best = s
        return best


sol = Solution()
for nums, expected in examples:
    print(sol.minElement(nums), "kỳ vọng", expected)

1 kỳ vọng 1
1 kỳ vọng 1
10 kỳ vọng 10


## 10. Vì sao thuật toán đúng

- **Tổng chữ số đúng:** với `x` có chữ số `d_k ... d_1 d_0`, ta có `x = 10·(x // 10) + (x % 10)`. Mỗi vòng cộng `x % 10` (chữ số cuối) rồi thay `x` bằng `x // 10` (chữ số còn lại). Sau đúng `k + 1` vòng `x` về `0` và tổng là `d_0 + d_1 + ... + d_k`.
- **Vòng lặp dừng:** `x` là số nguyên không âm và **giảm hẳn** sau mỗi vòng (`x // 10 < x` khi `x > 0`) ⇒ chắc chắn về `0`.
- **Đúng thứ tự công việc:** ta thay **từng** phần tử (tính tổng chữ số) rồi mới so với `best`, chứ không lấy min của số gốc.
- **Min đúng:** `best` luôn là min của các tổng chữ số đã xử lý (bất biến); khởi tạo `10 ** 9` lớn hơn mọi tổng nên phần tử đầu tiên chắc chắn thay thế nó; mảng không rỗng (`n >= 1`) nên `best` được cập nhật ít nhất một lần.
- **Cận dưới:** `x >= 1` ⇒ tổng chữ số `>= 1`, nên đáp án thuộc `[1, 36]` (tổng lớn nhất của số `<= 10⁴` là `9999 -> 36`). Dùng để kiểm tra nhanh kết quả hợp lý.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (`n <= 100`) và `d` là số chữ số lớn nhất (`d <= 5` vì `x <= 10⁴`).

- **Cách 1:** `O(n · d)` thời gian, `O(n)` bộ nhớ (mảng `replaced`).
- **Cách 2, 3, 4:** `O(n · d)` thời gian, `O(1)` bộ nhớ. Vì `d` bị chặn bởi hằng số `5` nên có thể nói **`O(n)`**. Về tổng quát `d = O(log₁₀ max(nums))`.
- **Cách 5:** `O(V + n)` thời gian với `V = max(nums)`, `O(V)` bộ nhớ.
- Cận dưới: phải đọc mọi phần tử ⇒ `Ω(n)`. Cách 2/3/4 đạt mức này (theo `n`).

## 12. Bộ test

Ta gom các cách vào `impls` rồi so sánh với kỳ vọng. Để kiểm chứng **độc lập**, ta thêm `reference(nums)` dùng định nghĩa thô nhất (đổi chuỗi, cộng, lấy min) và hàm `is_valid_answer(nums, ans)` kiểm tra: đáp án là **tổng chữ số của một phần tử nào đó** và **không lớn hơn** tổng chữ số của mọi phần tử.

In [19]:
def reference(nums):
    return min(sum(int(c) for c in str(x)) for x in nums)


def is_valid_answer(nums, ans):
    sums = [sum(int(c) for c in str(x)) for x in nums]
    return ans in sums and all(ans <= s for s in sums) and 1 <= ans <= 36


sol = Solution()
impls = {
    "baseline": min_element_baseline,
    "math": min_element_math,
    "oneliner": min_element_oneliner,
    "early": min_element_early,
    "table": min_element_table,
    "sol2": Solution2().minElement,
    "sol3": Solution3().minElement,
    "sol4": Solution4().minElement,
    "sol5": Solution5().minElement,
    "leetcode": sol.minElement,
}


def check_all(nums, expected, label):
    for key, fn in impls.items():
        actual = fn(list(nums))
        assert actual == expected, (label, key, nums, actual, expected)
        assert is_valid_answer(nums, actual), (label, key)
    assert reference(nums) == expected
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

Một phần tử, số lớn nhất `10000` (tổng chữ số `1`), `9999` (tổng chữ số lớn nhất `36`), số có chữ số `0` ở giữa, nhiều phần tử bằng nhau, và số nhỏ nhất ban đầu **không** cho tổng chữ số nhỏ nhất (đều tự dựng, hợp lệ theo ràng buộc).

In [20]:
check_all([1], 1, "một phần tử nhỏ nhất")
check_all([10000], 1, "một phần tử lớn nhất (10000 -> 1)")
check_all([9999], 36, "tổng chữ số lớn nhất của một số <= 10^4")
check_all([9999, 9999, 9999], 36, "mọi phần tử bằng nhau")
check_all([1000, 100, 10, 1], 1, "mọi tổng đều bằng 1")
check_all([19, 100], 1, "số nhỏ nhất (19) KHÔNG cho tổng nhỏ nhất (100 -> 1)")
check_all([909, 99, 18], 9, "chữ số 0 ở giữa: 909 -> 18, 99 -> 18, 18 -> 9")
check_all([5, 50, 500, 5000], 5, "các số 5, 50, 500, 5000 đều có tổng 5")

PASS một phần tử nhỏ nhất
PASS một phần tử lớn nhất (10000 -> 1)
PASS tổng chữ số lớn nhất của một số <= 10^4
PASS mọi phần tử bằng nhau
PASS mọi tổng đều bằng 1
PASS số nhỏ nhất (19) KHÔNG cho tổng nhỏ nhất (100 -> 1)
PASS chữ số 0 ở giữa: 909 -> 18, 99 -> 18, 18 -> 9
PASS các số 5, 50, 500, 5000 đều có tổng 5


Một mẹo tự kiểm tra bộ test: cố ý đưa **kỳ vọng sai** để chắc rằng `check_all` thực sự **bắt được lỗi** (nếu nó vẫn "PASS" thì bộ test vô dụng). Với `[9998, 9997]` ta có `9998 -> 35`, `9997 -> 34` nên đáp án đúng là `34`.

In [21]:
try:
    check_all([9998, 9997], 35, "ca cố ý sai")
except AssertionError:
    print("OK: assert đã bắt được kỳ vọng sai (35 sai, đúng là 34)")
check_all([9998, 9997], 34, "9998 -> 35, 9997 -> 34")

OK: assert đã bắt được kỳ vọng sai (35 sai, đúng là 34)
PASS 9998 -> 35, 9997 -> 34


### 12.2. Phản ví dụ cho các cách làm sai

Ba cách "nghe hợp lý" nhưng sai: (a) **tổng chữ số của số nhỏ nhất**, (b) **số nhỏ nhất của mảng** (quên thay), (c) khởi tạo **`best = 0`**.

In [22]:
def digit_sum(x):
    return sum(int(c) for c in str(x))

def wrong_digit_sum_of_min(nums):
    return digit_sum(min(nums))

def wrong_forget_replace(nums):
    return min(nums)

def wrong_best_zero(nums):
    best = 0
    for x in nums:
        best = min(best, digit_sum(x))
    return best


for name, wrong in [("tổng chữ số của min", wrong_digit_sum_of_min), ("quên thay", wrong_forget_replace), ("best = 0", wrong_best_zero)]:
    for nums in ([19, 100], [999, 19, 199], [10, 12, 13, 14]):
        if wrong(nums) != reference(nums):
            print(f"cách '{name}' sai trên {nums}: trả {wrong(nums)}, đúng là {reference(nums)}")
            break
    else:
        raise AssertionError(name)

cách 'tổng chữ số của min' sai trên [19, 100]: trả 10, đúng là 1
cách 'quên thay' sai trên [19, 100]: trả 19, đúng là 1
cách 'best = 0' sai trên [19, 100]: trả 0, đúng là 1


### 12.3. Vét cạn mọi giá trị `1..10 000` và đối chiếu ngẫu nhiên (có seed)

(1) Với **mọi** `x` từ `1` đến `10 000`: các cách tính tổng chữ số (chuỗi, `%`/`//`, bảng DP) phải **trùng nhau** và `Solution().minElement([x])` bằng tổng chữ số đó. (2) `5000` mảng ngẫu nhiên đúng ràng buộc (`1 <= n <= 100`, `1 <= x <= 10⁴`, seed cố định), đối chiếu mọi cách với `reference` và `is_valid_answer`.

In [23]:
for x in range(1, 10001):
    a = digit_sum_str(x)
    assert a == digit_sum_math(x) == ds[x] == digit_sum(x), x
    assert sol.minElement([x]) == a, x
print("PASS vét cạn 10000 giá trị: các cách tính tổng chữ số trùng nhau")
assert ds[0] == 0 and max(ds[1:]) == 36 and min(ds[1:]) == 1

rng = random.Random(3300)
for trial in range(5000):
    n = rng.choice([1, 2, 3, 5, 10, 50, 99, 100])
    hi = 10 if trial % 3 == 0 else (200 if trial % 3 == 1 else 10 ** 4)
    nums = [rng.randint(1, hi) for _ in range(n)]
    assert 1 <= len(nums) <= 100 and all(1 <= x <= 10 ** 4 for x in nums)
    ref = reference(nums)
    for key, fn in impls.items():
        got = fn(list(nums))
        assert got == ref, (key, nums, got, ref)
        assert is_valid_answer(nums, got), (key, nums)
print("PASS 5000 mảng ngẫu nhiên đúng ràng buộc")

PASS vét cạn 10000 giá trị: các cách tính tổng chữ số trùng nhau


PASS 5000 mảng ngẫu nhiên đúng ràng buộc


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Lấy tổng chữ số của số nhỏ nhất (thay sai thứ tự công việc)

In [24]:
def sai_min_truoc(nums):
    smallest = min(nums)                     # lấy min trước...
    return sum(int(c) for c in str(smallest))  # ...rồi mới tính tổng chữ số

print(sai_min_truoc([19, 100]), "(đúng là 1)")       # 10
print(Solution().minElement([19, 100]))

10 (đúng là 1)
1


**Vì sao sai:** đề yêu cầu **thay hết rồi mới lấy min**. Số nhỏ nhất ban đầu (`19`) không nhất thiết có tổng chữ số nhỏ nhất (`100` có tổng `1`). Phải tính tổng chữ số của **mọi** phần tử.

### Lỗi 2: `sum(str(x))` — cộng thẳng các ký tự (`TypeError`)

In [25]:
def sai_sum_chuoi(x):
    return sum(str(x))           # sum cần số, nhưng đây là các ký tự

try:
    sai_sum_chuoi(1234)
except TypeError as e:
    print("TypeError:", e)
print(sum(int(c) for c in str(1234)))     # đúng: 10

TypeError: unsupported operand type(s) for +: 'int' and 'str'
10


**Vì sao sai:** `str(1234)` cho `"1234"`, duyệt ra các **ký tự** `"1"`, `"2"`, ...; `sum` cố cộng chúng với `0` nên lỗi. Phải đổi từng ký tự về số bằng `int`: `sum(int(c) for c in str(x))` hoặc `sum(map(int, str(x)))`.

### Lỗi 3: Quên `x //= 10` — vòng lặp vô hạn

Ta **không** chạy vòng vô hạn thật; thay vào đó thêm một bộ đếm an toàn để dừng sau vài vòng và cho thấy `x` không hề thay đổi.

In [26]:
def sai_quen_chia(x, gioi_han=5):
    total = 0
    vong = 0
    while x > 0:
        total += x % 10
        # thiếu: x //= 10  -> x không đổi, điều kiện x > 0 luôn đúng
        vong += 1
        if vong >= gioi_han:                 # CHỈ để demo: thoát cưỡng bức
            print(f"  đã lặp {vong} vòng mà x vẫn = {x}, total = {total} -> vòng lặp sẽ chạy mãi")
            break
    return total

sai_quen_chia(123)

  đã lặp 5 vòng mà x vẫn = 123, total = 15 -> vòng lặp sẽ chạy mãi


15

**Vì sao sai:** điều kiện `x > 0` chỉ sai khi `x` giảm về `0`. Nếu không cập nhật `x` thì vòng lặp **không bao giờ dừng** (LeetCode báo `Time Limit Exceeded`). Quy tắc: mỗi vòng `while` phải làm cho điều kiện **tiến gần** đến sai.

### Lỗi 4: Dùng `/` thay `//` khi tách chữ số

In [27]:
def sai_chia_thuc(x, gioi_han=6):
    total = 0
    vong = 0
    while x > 0:
        total += x % 10
        x = x / 10                            # sai: ra số thực, không về 0 ngay
        vong += 1
        print(f"  vòng {vong}: x = {x}, total = {total}")
        if vong >= gioi_han:
            print("  ... x vẫn > 0 (số thực rất nhỏ nhưng dương), vòng lặp còn kéo dài rất nhiều vòng")
            break
    return total

sai_chia_thuc(123)

  vòng 1: x = 12.3, total = 3
  vòng 2: x = 1.23, total = 5.300000000000001
  vòng 3: x = 0.123, total = 6.530000000000001
  vòng 4: x = 0.0123, total = 6.653000000000001
  vòng 5: x = 0.00123, total = 6.665300000000001
  vòng 6: x = 0.000123, total = 6.666530000000001
  ... x vẫn > 0 (số thực rất nhỏ nhưng dương), vòng lặp còn kéo dài rất nhiều vòng


6.666530000000001

**Vì sao sai:** `123 / 10 = 12.3`, rồi `1.23`, `0.123`, ... — **không bao giờ đúng bằng `0`** trong thời gian hợp lý, và `x % 10` trên số thực cho phần lẻ như `2.3`, làm tổng sai. Hãy dùng `//` (chia nguyên).

### Lỗi 5: Khởi tạo `best = 0`

In [28]:
def sai_best_0(nums):
    best = 0                                  # 0 nhỏ hơn mọi tổng chữ số (>= 1) -> không bao giờ cập nhật
    for x in nums:
        s = sum(int(c) for c in str(x))
        if s < best:
            best = s
    return best

print(sai_best_0([10, 12, 13, 14]), "(đúng là 1)")      # 0

0 (đúng là 1)


**Vì sao sai:** mọi tổng chữ số đều `>= 1 > 0` nên điều kiện `s < best` không bao giờ đúng và kết quả luôn `0`. Khi tìm **min**, khởi tạo bằng **phần tử đầu tiên** hoặc một **số rất lớn** (`float("inf")`, `10 ** 9`); khi tìm **max**, khởi tạo bằng phần tử đầu hoặc số rất nhỏ.

**Code đúng (nhắc lại):** mỗi `x`: tính tổng chữ số `s` bằng `% 10`/`// 10`, rồi `best = min(best, s)`.

In [29]:
def dung(nums):
    best = 10 ** 9
    for x in nums:
        s = 0
        while x > 0:
            s += x % 10
            x //= 10
        best = min(best, s)
    return best

for nums, expected in examples:
    assert dung(nums) == expected
print("PASS code đúng")

PASS code đúng


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Tổng các chữ số**" ⇒ hai cách chuẩn: `sum(map(int, str(x)))` (ngắn) hoặc vòng `while x: s += x % 10; x //= 10` (kỹ thuật toán, dùng lại cho đảo số, palindrome số, đếm chữ số...).
- "**Thay hết rồi mới lấy min**" ⇒ tách thành hai việc: biến đổi từng phần tử rồi `min(...)`. Đừng lấy min trên dữ liệu gốc.
- Biến đổi + tìm min một lượt ⇒ `min(f(x) for x in nums)`; dùng generator để khỏi lưu danh sách.
- Khi tìm min/max bằng vòng lặp, **khởi tạo đúng** (`phần tử đầu` hoặc `inf`), nhớ xét mảng 1 phần tử.
- Biết **cận dưới**: tổng chữ số của số nguyên dương `>= 1`; có thể thoát sớm khi đã đạt cận.
- Mỗi vòng `while` phải làm điều kiện tiến đến sai (`x //= 10`) — hãy kiểm tra dòng này **trước tiên** nếu gặp Time Limit Exceeded.

## 15. Biến thể của bài

1. **Tổng chữ số lớn nhất** thay vì nhỏ nhất.
2. **Chỉ số** của phần tử có tổng chữ số nhỏ nhất (phần tử đầu tiên nếu hòa).
3. **Số gốc** có tổng chữ số nhỏ nhất (nếu hòa thì lấy số nhỏ nhất) — bài hay gặp hơn trên thực tế.
4. **Căn số số học** (digital root): lặp thay bằng tổng chữ số cho đến khi còn **một chữ số** (LeetCode 258), có công thức `1 + (x - 1) % 9` cho `x >= 1`.
5. **Tích các chữ số**, hoặc tổng chữ số trong **hệ cơ số khác** (ví dụ nhị phân, bát phân).

In [30]:
def max_digit_sum(nums):
    return max(sum(map(int, str(x))) for x in nums)


def index_of_min_digit_sum(nums):
    sums = [sum(map(int, str(x))) for x in nums]
    return sums.index(min(sums))                     # index trả vị trí đầu tiên khi hòa


def original_with_min_digit_sum(nums):
    return min(nums, key=lambda x: (sum(map(int, str(x))), x))


def digital_root(x):
    while x >= 10:
        x = sum(map(int, str(x)))
    return x


def digit_sum_base(x, base):
    total = 0
    while x > 0:
        total += x % base
        x //= base
    return total


print(max_digit_sum([999, 19, 199]))                 # 27
print(index_of_min_digit_sum([999, 19, 199]))        # 1
print(original_with_min_digit_sum([19, 100, 1000]))  # 100  (hòa tổng 1 -> số nhỏ hơn)
print(digital_root(9875), 1 + (9875 - 1) % 9)         # 2 2
print(digit_sum_base(29, 2), bin(29).count("1"))      # 4 4
print(Solution().minElement([999, 19, 199]))          # 10

27
1
100
2 2
4 4
10


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng mảng mới rồi tìm min | `replaced` + vòng tìm min | `O(n · d)` | `O(n)` | bám sát đề, dễ đọc cho người mới |
| 2. `% 10` và `// 10` | tách chữ số bằng toán, giữ `best` | `O(n · d)` | `O(1)` | **nên nộp**, kỹ thuật dùng lại nhiều nơi |
| 3. `min` + generator + `sum(map(int, str))` | một dòng | `O(n · d)` | `O(1)` | gọn, quen tay Python |
| 4. `divmod` + thoát sớm | dừng khi `best == 1` | `O(n · d)` (tệ nhất) | `O(1)` | tối ưu nhỏ nhờ cận dưới |
| 5. Bảng DP `ds[x] = ds[x // 10] + x % 10` | tra bảng | `O(V + n)` | `O(V)` | rất nhiều truy vấn trên miền giá trị nhỏ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `1234 % 10`, `1234 // 10`, `sum(map(int, "405"))`, `Solution().minElement([10, 12, 13, 14])`, `Solution().minElement([19, 100])`.

**Bài 2 (tính tay).** Với `nums = [58, 9, 307, 1999]` và `nums = [64, 1000, 91]`: tính tổng chữ số của từng phần tử và cho đáp án.

**Bài 3 (điền TODO).** Hoàn thành `digit_sum_todo(x)` (tổng chữ số bằng `%` và `//`) và `min_element_todo(nums)`.

In [31]:
def digit_sum_todo(x):
    total = 0
    # TODO 1: while x > 0: cộng x % 10 vào total
    # TODO 2: bỏ chữ số cuối bằng x //= 10 (đừng quên, nếu không vòng lặp chạy mãi!)
    return total


def min_element_todo(nums):
    # TODO 3: trả về min của digit_sum_todo(x) với mọi x trong nums
    pass

# KHÔNG gọi các hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `min_element_memo(nums)` dùng **`dict` nhớ kết quả** (`memo`): trước khi tính tổng chữ số của `x`, kiểm tra `x` đã có trong `memo` chưa; nếu có thì dùng lại. (Hữu ích khi mảng có nhiều số trùng nhau.)

**Bài 5 (biến thể / mini challenge).** (a) Viết `digital_root_formula(x)` dùng **công thức** `1 + (x - 1) % 9` (với `x >= 1`) và kiểm tra nó trùng với phiên bản lặp cho mọi `x` từ `1` đến `10 000`. (b) Viết `group_by_digit_sum(nums)` trả `dict` ánh xạ **tổng chữ số → danh sách các số gốc (tăng dần, không trùng)**.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `4`, `123`, `9`, `1`, `1`. Bài 3: `while x > 0: total += x % 10; x //= 10`, rồi `min(digit_sum_todo(x) for x in nums)`.
- **Mức 2 (vừa):** Bài 2: `[58, 9, 307, 1999]` ⇒ `13, 9, 10, 28` ⇒ `9`; `[64, 1000, 91]` ⇒ `10, 1, 10` ⇒ `1`. Bài 4: `if x not in memo: memo[x] = digit_sum(x)`.
- **Mức 3 (sâu):** Bài 5(a): thử vài số tay (`9875 -> 29 -> 11 -> 2` và `1 + 9874 % 9 = 1 + 1 = 2`), rồi `assert` cho cả khoảng `1..10000`. 5(b): dùng `dict.setdefault(s, set()).add(x)` rồi đổi mỗi `set` thành `sorted(...)`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert 1234 % 10 == 4
assert 1234 // 10 == 123
assert sum(map(int, "405")) == 9
assert Solution().minElement([10, 12, 13, 14]) == 1
assert Solution().minElement([19, 100]) == 1

# Bài 2
assert Solution().minElement([58, 9, 307, 1999]) == 9
assert Solution().minElement([64, 1000, 91]) == 1


# Bài 3
def digit_sum_todo_ans(x):
    total = 0
    while x > 0:
        total += x % 10
        x //= 10
    return total


def min_element_todo_ans(nums):
    return min(digit_sum_todo_ans(x) for x in nums)


for nums, expected in examples:
    assert min_element_todo_ans(nums) == expected
assert digit_sum_todo_ans(1999) == 28 and digit_sum_todo_ans(10000) == 1


# Bài 4
def min_element_memo(nums):
    memo = {}
    best = 10 ** 9
    for x in nums:
        if x not in memo:
            memo[x] = digit_sum_todo_ans(x)
        best = min(best, memo[x])
    return best


for nums, expected in examples:
    assert min_element_memo(nums) == expected
assert min_element_memo([777] * 100) == 21


# Bài 5(a)
def digital_root_formula(x):
    return 1 + (x - 1) % 9


for x in range(1, 10001):
    assert digital_root_formula(x) == digital_root(x), x


# Bài 5(b)
def group_by_digit_sum(nums):
    groups = {}
    for x in nums:
        groups.setdefault(digit_sum_todo_ans(x), set()).add(x)
    return {s: sorted(vs) for s, vs in groups.items()}


assert group_by_digit_sum([10, 1, 100, 19, 91, 19]) == {1: [1, 10, 100], 10: [19, 91]}
assert group_by_digit_sum([5]) == {5: [5]}
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tổng chữ số bằng toán, một lượt giữ `best`)

```python
class Solution:
    def minElement(self, nums: List[int]) -> int:
        best = 10 ** 9
        for x in nums:
            s = 0
            while x > 0:
                s += x % 10
                x //= 10
            best = min(best, s)
        return best
```

**Checklist**

- [ ] Đã thay **mọi** phần tử bằng tổng chữ số **trước** khi lấy min (không lấy min của số gốc)?
- [ ] Vòng `while x > 0` có `x //= 10` (dùng `//`, không phải `/`)?
- [ ] `best` khởi tạo đúng (phần tử đầu / `10 ** 9`), **không** phải `0`?
- [ ] Nếu dùng `str`: đã `int(c)` từng ký tự (không `sum(str(x))`)?
- [ ] Đã thử ba ví dụ đề (`1`, `1`, `10`), một phần tử, `10000` (đáp án `1`), và `[19, 100]` (đáp án `1`)?
- [ ] Skeleton đúng `class Solution` / `def minElement(self, nums: List[int]) -> int`?